# TripPulse — Week 6
## Data Quality: Silver Candidate to Trusted Silver and Quarantine

**Notebook path:** `notebooks/04_data_quality_checks.ipynb`  
**Technology:** Databricks Free Edition | Spark SQL | Delta tables

> **Week 6 Goal:** Test every Week-5 Candidate record against the approved TripPulse DQ rules. Records that pass become **Trusted Silver**. Records that fail are retained in **Quarantine with all failure reasons**. No record is silently deleted.

This notebook follows the structure and teaching style of the supplied PageLoop Week-6 example, but the table names, fields and rules below are TripPulse-specific.


## 1. Outcome first - what will TripPulse produce?

Week 5 prepared typed, standardised **Silver Candidate** tables. Candidate means *ready for quality assessment*; it does not yet mean trusted.

| Week-5 input | Week-6 Trusted output | Week-6 Quarantine output |
|---|---|---|
| `silver_zones_candidate` | `trusted_silver_zones` | `quarantine_zones` |
| `silver_drivers_candidate` | `trusted_silver_drivers` | `quarantine_drivers` |
| `silver_trips_candidate` | `trusted_silver_trips` | `quarantine_trips` |
| `silver_payments_candidate` | `trusted_silver_payments` | `quarantine_payments` |

The current Week-5 implementation contains these four batch Candidate tables. Ride-request event Candidate processing belongs to the later controlled streaming work; this notebook therefore documents DQ-STR-001 to DQ-STR-004 as **preparation/classification rules** rather than fabricating streaming results.

The non-negotiable proof for every batch entity is:

`Candidate rows = Trusted rows + Quarantine rows`

**Success standard:** every Candidate physical record appears exactly once on one side of the split, and every quarantined row explains why it failed.


## 2. Week 6 in one minute

Think of an airport security queue:

| TripPulse concept | Airport analogy | Meaning |
|---|---|---|
| Silver Candidate | passengers waiting for checks | prepared records awaiting DQ |
| DQ rulebook | documented security checks | approved conditions, not personal guesses |
| Trusted Silver | cleared passengers | records that passed every governing rule |
| Quarantine | secondary inspection | records retained with clear failure reasons |
| Diagnostic profile | an observation | useful to investigate, but not automatically a rejection |

> A value can be unusual without being invalid. Quarantine only when an approved TripPulse rule fails.


## 3. What you will learn and do

By the end of this notebook, you will be able to:

1. explain the difference between profiling, diagnostics and governing DQ rules;
2. implement completeness, uniqueness, reference, range, chronology, status and lineage checks;
3. use readable `CASE WHEN` statements to mark each rule `PASS` or `FAIL`;
4. retain every applicable failure reason when one record breaks several rules;
5. route records into the exact TripPulse Trusted Silver and Quarantine tables;
6. reconcile counts and physical-record membership;
7. evaluate payment rules at payment-attempt group grain without collapsing attempts;
8. explain the four streaming-preparation rules without pretending Week-10 streaming has already run.

**Coding approach:** one small step at a time — prepare, check, inspect, summarise, route and prove.


## 4. Today's seven-action journey

| Action | What you do | Evidence produced |
|---:|---|---|
| 1 | Confirm the Week-5 Candidate handoff | tables, counts and lineage fields |
| 2 | Read the TripPulse DQ rulebook | rule IDs, meanings and severity |
| 3 | Apply one rule family at a time | visible `PASS/FAIL` columns |
| 4 | Capture all failures for each row | failure count and readable reasons |
| 5 | Route each batch entity | Trusted and Quarantine Delta tables |
| 6 | Prove no silent loss | count and physical-record reconciliation |
| 7 | Explain rerun/replay and streaming preparation | repeat-run and later-stream evidence path |

Do not jump directly to table creation. The inspection steps are where you learn to explain the result.


## 5. Three levels of quality checking

| Level | Question | TripPulse example | Changes route? |
|---|---|---|---|
| Profile | What values and patterns exist? | distribution of `trip_status` | No |
| Diagnostic | Does something deserve investigation? | unusual fare variance | No, until an approved threshold exists |
| Governing rule | Does the record violate an approved condition? | `dropoff_ts <= pickup_ts` for a completed trip | Yes |

**Why this matters:** Week 6 should not turn every unusual value into a failure. It should also not allow an explicitly invalid record into Trusted Silver.


## 6. TripPulse DQ rulebook - written for humans

### 6.1 Zone rule

| Rule ID | Record fails when... | Why it matters | Severity |
|---|---|---|---|
| `DQ-ZON-001` | key/pattern/uniqueness, zone name/category/city/demand/active flag/effective date violates the approved contract | invalid zones break dependent joins | Critical |

### 6.2 Driver rule

| Rule ID | Record fails when... | Why it matters | Severity |
|---|---|---|---|
| `DQ-DRV-001` | driver key/pattern/uniqueness, home-zone reference, vehicle/service/status/rating/count/date/version violates the approved contract | invalid drivers corrupt eligibility and performance analysis | Critical |

### 6.3 Trip rules

| Rule ID | Record fails when... | Severity |
|---|---|---|
| `DQ-TRIP-001` | `trip_id` is missing/blank, wrong `TRP-YYYYMMDD-NNNNNN` pattern, or duplicated | Critical |
| `DQ-TRIP-002` | pickup/drop-off zone or present driver does not resolve; driver is missing for accepted/started/completed lifecycle | Critical |
| `DQ-TRIP-003` | lifecycle violates `request <= accept <= pickup < dropoff` where applicable, or required timestamps are missing | Critical |
| `DQ-TRIP-004` | status contradicts driver/cancel/drop-off/cancellation-reason fields or is unsupported | Major |
| `DQ-TRIP-005` | assigned driver cannot satisfy trip service because of driver status/service compatibility | Major |
| `DQ-TRIP-006` | distance is outside the approved range or conflicts with lifecycle presence rules | Major |
| `DQ-TRIP-007` | fare/surge is outside approved ranges or conditional fare fields conflict with status | Major |
| `DQ-TRIP-008` | request is outside Jan-Mar 2026, lineage timing is incoherent, or required Bronze lineage is missing | Major |

### 6.4 Payment rules

| Rule ID | Record fails when... | Severity |
|---|---|---|
| `DQ-PAY-001` | payment key/reference is missing, wrong pattern/duplicate, or `trip_id` is missing/not found | Critical |
| `DQ-PAY-002` | attempt group violates attempt/status/reason/amount/time/final-attempt logic or final successful amount does not reconcile to completed-trip fare | Major |

### 6.5 Streaming-preparation rules

| Rule ID | Retained classification | Severity |
|---|---|---|
| `DQ-STR-001` | event identity/reference failure or duplicate | Critical |
| `DQ-STR-002` | event-time parsing/10-minute watermark classification, retaining late events | Major |
| `DQ-STR-003` | sequence/approved state-transition failure | Major |
| `DQ-STR-004` | schema v1.0/type/range/corrupt-payload failure; rescued payload retained | Critical |

**Dependency order:** zones → drivers using trusted zones → trips using trusted zones/drivers → payments using trips. The four streaming rules are documented here for later Week-10 execution.


## 7. Scenario gallery - predict before running

| Scenario | Expected route | Reason |
|---|---|---|
| Valid zone with approved category/date | Trusted | DQ-ZON-001 passes |
| Driver with unknown home zone | Quarantine | DQ-DRV-001 |
| Completed trip with drop-off before pickup | Quarantine | DQ-TRIP-003 |
| Completed trip with `cancel_ts` populated | Quarantine | DQ-TRIP-004 |
| Trip assigned to incompatible driver service | Quarantine | DQ-TRIP-005 |
| Completed trip with null actual distance | Quarantine | DQ-TRIP-006 |
| Trip with surge 3.50 | Quarantine | DQ-TRIP-007 |
| Payment points to unknown trip | Quarantine | DQ-PAY-001 |
| One payment trip group has two final attempts | Quarantine | DQ-PAY-002 |
| One row fails multiple rules | one Quarantine row with all rule IDs | evaluate every rule, not only the first |
| Late streaming event | retained/classified, not silently deleted | DQ-STR-002 |


## 8. Confirm the Week-5 handoff

Week 6 reads the outputs of Week 5. It does not rebuild Silver transformations.


In [0]:
%sql
USE CATALOG `TripPulse`;
USE SCHEMA `default`;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;

active_catalog,active_schema
trippulse,default


In [0]:
%sql
SHOW TABLES LIKE 'silver_*_candidate';

database,tableName,isTemporary
default,silver_drivers_candidate,false
default,silver_payments_candidate,false
default,silver_trips_candidate,false
default,silver_zones_candidate,false


In [0]:
%sql
SELECT 'zones' AS entity, COUNT(*) AS candidate_rows FROM silver_zones_candidate
UNION ALL SELECT 'drivers', COUNT(*) FROM silver_drivers_candidate
UNION ALL SELECT 'trips', COUNT(*) FROM silver_trips_candidate
UNION ALL SELECT 'payments', COUNT(*) FROM silver_payments_candidate;

entity,candidate_rows
zones,120
drivers,2800
trips,250875
payments,180315


In [0]:
%sql

SELECT *
FROM (
    SELECT
        'zones' AS entity,
        zone_id AS record_id,
        _source_file_name,
        _bronze_record_hash
    FROM silver_zones_candidate
    LIMIT 5
)

UNION ALL

SELECT *
FROM (
    SELECT
        'drivers' AS entity,
        driver_id AS record_id,
        _source_file_name,
        _bronze_record_hash
    FROM silver_drivers_candidate
    LIMIT 5
);

entity,record_id,_source_file_name,_bronze_record_hash
zones,ZON-001,zones.csv,4d47db40536551ced70bc60ef018d84ba61d3e5378d1e7c04d121b6949c4433d
zones,ZON-002,zones.csv,8c629d27a7072cda440fb389cd754ae0b4b3deac1c7294721f21c4b280547fa3
zones,ZON-003,zones.csv,2e4e78e53e350d53f6b573da7688133e5ae125917921d2c9342b7cf63eec480a
zones,ZON-004,zones.csv,ddd7959f3783ae1e8a0b296c28284f28038a94639c3ecbd28ef3dee2b1cf84f5
zones,ZON-005,zones.csv,2651e4564e048bd69617953ab7dccadb7356e0d422c89723aab8ad782d22a0ce
drivers,DRV-000001,drivers.json,0aae62247fcecc03004674e73cf981f5c82273dbd88c3d3dffa2c3f21a8a4c38
drivers,DRV-000002,drivers.json,7000ea23bc329518927575c5298c4274407b4af27bbf275c055e6baa8d365460
drivers,DRV-000003,drivers.json,547c43c700e84ca822ff1c43966175affb7f9f207f716ffdb099119b4d2a76cd
drivers,DRV-000004,drivers.json,5602a01ec941c545c2fd08f0690ebf786ef171bbc76fa2f185b20d876366f85c
drivers,DRV-000005,drivers.json,01c54bf0d88e22f3efa6e431449ea278b7c9f9efe1954be45adc6340ca522302


**Checkpoint:** explain the difference between a business key (`trip_id`, `payment_id`, etc.) and `_bronze_record_hash`, which traces the physical Bronze row.


## 9. Learn the code pattern with one rule

The main notebook uses the familiar pattern:

```sql
CASE WHEN invalid_condition THEN 'FAIL' ELSE 'PASS' END
```

Start with a required Trip field. This cell demonstrates the rule only; it does not write a final table.


In [0]:
%sql
SELECT trip_id,
       CASE WHEN trip_id IS NULL OR trim(trip_id) = '' THEN 'FAIL' ELSE 'PASS' END AS trip_key_check
FROM silver_trips_candidate
LIMIT 20;

trip_id,trip_key_check
TRP-20260127-000001,PASS
TRP-20260306-000002,PASS
TRP-20260219-000003,PASS
TRP-20260223-000004,PASS
TRP-20260312-000005,PASS
TRP-20260313-000006,PASS
TRP-20260228-000007,PASS
TRP-20260226-000008,PASS
TRP-20260129-000009,PASS
TRP-20260126-000010,PASS


**Read it aloud:** “When the Trip ID is missing or blank, mark FAIL; otherwise mark PASS.”


## 10. Build Zone DQ in small steps

### 10.1 Find duplicate zone IDs

Both physical rows sharing a duplicate business key fail the uniqueness rule; we do not arbitrarily keep one.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_zone_ids AS
SELECT zone_id, COUNT(*) AS occurrences
FROM silver_zones_candidate
WHERE zone_id IS NOT NULL AND trim(zone_id) <> ''
GROUP BY zone_id
HAVING COUNT(*) > 1;

SELECT * FROM duplicate_zone_ids ORDER BY occurrences DESC LIMIT 20;

zone_id,occurrences


### 10.2 Apply the Zone rule

`DQ-ZON-001` is one governing rule, but it contains several contract checks. The visible columns make each part explainable.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW zones_checked AS
SELECT z.*,
  CASE WHEN z.zone_id IS NULL OR trim(z.zone_id) = ''
             OR z.zone_id NOT RLIKE '^ZON-[0-9]{3}$'
             OR d.zone_id IS NOT NULL
             OR z.zone_name IS NULL OR trim(z.zone_name) = '' OR length(trim(z.zone_name)) > 80
             OR z.zone_type NOT IN ('residential','commercial','transit_hub','education','mixed_use','airport')
             OR z.city_code <> 'TPC'
             OR z.demand_band NOT IN ('low','medium','high')
             OR z.is_active IS NULL
             OR z.effective_from IS NULL
             OR z.effective_from < DATE '2025-01-01'
             OR z.effective_from > DATE '2026-03-31'
       THEN 'FAIL' ELSE 'PASS' END AS zon001_check
FROM silver_zones_candidate z
LEFT JOIN duplicate_zone_ids d ON z.zone_id = d.zone_id;

SELECT zone_id, zone_name, zone_type, city_code, demand_band, is_active, effective_from, zon001_check
FROM zones_checked LIMIT 20;

zone_id,zone_name,zone_type,city_code,demand_band,is_active,effective_from,zon001_check
ZON-001,TPC Zone 001,residential,TPC,low,true,2025-01-01,PASS
ZON-002,TPC Zone 002,commercial,TPC,medium,true,2025-01-02,PASS
ZON-003,TPC Zone 003,transit_hub,TPC,high,true,2025-01-03,PASS
ZON-004,TPC Zone 004,education,TPC,low,true,2025-01-04,PASS
ZON-005,TPC Zone 005,mixed_use,TPC,medium,true,2025-01-05,PASS
ZON-006,TPC Zone 006,airport,TPC,high,true,2025-01-06,PASS
ZON-007,TPC Zone 007,residential,TPC,low,true,2025-01-07,PASS
ZON-008,TPC Zone 008,commercial,TPC,medium,true,2025-01-08,PASS
ZON-009,TPC Zone 009,transit_hub,TPC,high,true,2025-01-09,PASS
ZON-010,TPC Zone 010,education,TPC,low,true,2025-01-10,PASS


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW zones_routed AS
SELECT *,
  CASE WHEN zon001_check = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN zon001_check = 'FAIL' THEN 'DQ-ZON-001' ELSE '[]' END AS failed_rule_ids,
  CASE WHEN zon001_check = 'FAIL' THEN 'ZONE_REFERENCE_INVALID' ELSE '' END AS failure_reasons,
  CASE WHEN zon001_check = 'FAIL' THEN 'Critical' ELSE 'NONE' END AS severity,
  CASE WHEN zon001_check = 'FAIL' THEN 'zone_id, zone_name, zone_type, city_code, demand_band, is_active, effective_from' ELSE '' END AS affected_field
FROM zones_checked;

SELECT zone_id, zon001_check, failed_rule_ids, dq_status, severity FROM zones_routed LIMIT 20;

zone_id,zon001_check,failed_rule_ids,dq_status,severity
ZON-001,PASS,[],PASS,NONE
ZON-002,PASS,[],PASS,NONE
ZON-003,PASS,[],PASS,NONE
ZON-004,PASS,[],PASS,NONE
ZON-005,PASS,[],PASS,NONE
ZON-006,PASS,[],PASS,NONE
ZON-007,PASS,[],PASS,NONE
ZON-008,PASS,[],PASS,NONE
ZON-009,PASS,[],PASS,NONE
ZON-010,PASS,[],PASS,NONE


In [0]:
%sql

CREATE OR REPLACE TABLE trusted_silver_zones
USING DELTA
AS
SELECT *
FROM zones_routed
WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_zones
USING DELTA
AS
SELECT *
FROM zones_routed
WHERE dq_status = 'FAIL';

num_affected_rows,num_inserted_rows


## 11. Build Driver DQ

### 11.1 Driver rule and trusted-zone dependency

Drivers must resolve their `home_zone_id` to an **active** trusted zone. This is why the approved dependency order validates zones before drivers.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_driver_ids AS
SELECT driver_id, COUNT(*) AS occurrences
FROM silver_drivers_candidate
WHERE driver_id IS NOT NULL AND trim(driver_id) <> ''
GROUP BY driver_id HAVING COUNT(*) > 1;

CREATE OR REPLACE TEMP VIEW drivers_checked AS
SELECT d.*,
  CASE WHEN d.driver_id IS NULL OR trim(d.driver_id) = ''
             OR d.driver_id NOT RLIKE '^DRV-[0-9]{6}$'
             OR dup.driver_id IS NOT NULL
             OR d.home_zone_id IS NULL OR z.zone_id IS NULL OR z.is_active <> TRUE
             OR d.onboard_date IS NULL OR d.onboard_date < DATE '2023-01-01' OR d.onboard_date > DATE '2026-03-31'
             OR d.vehicle_type NOT IN ('bike','auto','mini','sedan')
             OR d.service_type NOT IN ('bike_taxi','auto','mini','sedan')
             OR d.driver_status NOT IN ('active','inactive','suspended')
             OR (d.rating IS NOT NULL AND (d.rating < 1.00 OR d.rating > 5.00))
             OR d.lifetime_completed_trips IS NULL OR d.lifetime_completed_trips < 0 OR d.lifetime_completed_trips > 50000
             OR d.last_status_update_ts IS NULL
             OR d.last_status_update_ts < TIMESTAMP '2025-01-01 00:00:00'
             OR d.last_status_update_ts > TIMESTAMP '2026-03-31 23:59:59'
             OR d.source_record_version IS NULL OR d.source_record_version <= 0
       THEN 'FAIL' ELSE 'PASS' END AS drv001_check
FROM silver_drivers_candidate d
LEFT JOIN duplicate_driver_ids dup ON d.driver_id = dup.driver_id
LEFT JOIN trusted_silver_zones z ON d.home_zone_id = z.zone_id;

SELECT driver_id, home_zone_id, vehicle_type, service_type, driver_status, rating, drv001_check
FROM drivers_checked LIMIT 20;

driver_id,home_zone_id,vehicle_type,service_type,driver_status,rating,drv001_check
DRV-000001,ZON-999,sedan,sedan,active,3.96,FAIL
DRV-000002,ZON-999,auto,auto,active,4.78,FAIL
DRV-000003,ZON-999,sedan,sedan,active,3.36,FAIL
DRV-000004,ZON-999,auto,auto,active,4.64,FAIL
DRV-000005,ZON-999,auto,auto,inactive,4.09,FAIL
DRV-000006,ZON-999,bike,bike_taxi,active,3.64,FAIL
DRV-000007,ZON-999,bike,bike_taxi,active,3.47,FAIL
DRV-000008,ZON-112,bike,bike_taxi,active,3.67,PASS
DRV-000009,ZON-037,auto,auto,active,4.78,PASS
DRV-000010,ZON-071,auto,auto,active,4.93,PASS


**Important:** the driver rule is evaluated after the Zone Trusted output exists. If your Zone DQ has quarantined a zone, a driver pointing to that zone cannot be treated as trusted.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW drivers_routed AS
SELECT *,
  CASE WHEN drv001_check = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN drv001_check = 'FAIL' THEN 'DQ-DRV-001' ELSE '[]' END AS failed_rule_ids,
  CASE WHEN drv001_check = 'FAIL' THEN 'DRIVER_REFERENCE_INVALID' ELSE '' END AS failure_reasons,
  CASE WHEN drv001_check = 'FAIL' THEN 'Critical' ELSE 'NONE' END AS severity,
  CASE WHEN drv001_check = 'FAIL' THEN 'driver_id, home_zone_id, onboard_date, vehicle_type, service_type, driver_status, rating, lifetime_completed_trips, last_status_update_ts, source_record_version' ELSE '' END AS affected_field
FROM drivers_checked;

SELECT driver_id, drv001_check, failed_rule_ids, dq_status, severity FROM drivers_routed LIMIT 20;

driver_id,drv001_check,failed_rule_ids,dq_status,severity
DRV-000001,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000002,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000003,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000004,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000005,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000006,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000007,FAIL,DQ-DRV-001,FAIL,Critical
DRV-000008,PASS,[],PASS,NONE
DRV-000009,PASS,[],PASS,NONE
DRV-000010,PASS,[],PASS,NONE


## 12. Build Trip DQ - complete worked example

Trips receive the deepest walkthrough because they combine keys, references, lifecycle timestamps, status conditions, driver compatibility, ranges and lineage.

### 12.1 Find duplicate Trip IDs


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_trip_ids AS
SELECT trip_id, COUNT(*) AS occurrences
FROM silver_trips_candidate
WHERE trip_id IS NOT NULL AND trim(trip_id) <> ''
GROUP BY trip_id HAVING COUNT(*) > 1;

SELECT * FROM duplicate_trip_ids ORDER BY occurrences DESC LIMIT 20;

trip_id,occurrences
TRP-20260219-006286,2
TRP-20260125-005486,2
TRP-20260114-006154,2
TRP-20260203-006225,2
TRP-20260222-004075,2
TRP-20260130-004765,2
TRP-20260219-003543,2
TRP-20260122-006124,2
TRP-20260227-003376,2
TRP-20260315-003163,2


In [0]:
%sql

CREATE OR REPLACE TABLE trusted_silver_drivers
USING DELTA
AS
SELECT *
FROM drivers_routed
WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_drivers
USING DELTA
AS
SELECT *
FROM drivers_routed
WHERE dq_status = 'FAIL';

num_affected_rows,num_inserted_rows


### 12.2 Apply identity and reference checks

References are checked against the accepted Zone and Driver dimensions produced by the earlier DQ stages.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW trips_reference_checked AS
SELECT t.*,
  CASE WHEN t.trip_id IS NULL OR trim(t.trip_id) = '' OR t.trip_id NOT RLIKE '^TRP-[0-9]{8}-[0-9]{6}$' OR dup.trip_id IS NOT NULL THEN 'FAIL' ELSE 'PASS' END AS trip001_check,
  CASE WHEN t.pickup_zone_id IS NULL OR pz.zone_id IS NULL OR pz.is_active <> TRUE
             OR t.dropoff_zone_id IS NULL OR dz.zone_id IS NULL OR dz.is_active <> TRUE
             OR (t.driver_id IS NOT NULL AND dr.driver_id IS NULL)
             OR (t.trip_status IN ('completed') AND (t.driver_id IS NULL OR dr.driver_id IS NULL))
       THEN 'FAIL' ELSE 'PASS' END AS trip002_check
FROM silver_trips_candidate t
LEFT JOIN duplicate_trip_ids dup ON t.trip_id = dup.trip_id
LEFT JOIN trusted_silver_zones pz ON t.pickup_zone_id = pz.zone_id
LEFT JOIN trusted_silver_zones dz ON t.dropoff_zone_id = dz.zone_id
LEFT JOIN trusted_silver_drivers dr ON t.driver_id = dr.driver_id;

SELECT trip_id, driver_id, pickup_zone_id, dropoff_zone_id, trip_status, trip001_check, trip002_check
FROM trips_reference_checked LIMIT 20;

trip_id,driver_id,pickup_zone_id,dropoff_zone_id,trip_status,trip001_check,trip002_check
TRP-20260127-000001,DRV-000620,ZON-035,ZON-016,completed,PASS,PASS
TRP-20260306-000002,DRV-002433,ZON-002,ZON-115,completed,PASS,PASS
TRP-20260219-000003,DRV-002305,ZON-078,ZON-106,cancelled_by_rider,PASS,PASS
TRP-20260223-000004,DRV-001102,ZON-022,ZON-009,completed,PASS,PASS
TRP-20260312-000005,DRV-000357,ZON-087,ZON-050,cancelled_by_driver,PASS,PASS
TRP-20260313-000006,null,ZON-054,ZON-090,unfulfilled,PASS,PASS
TRP-20260228-000007,DRV-002354,ZON-015,ZON-106,completed,PASS,PASS
TRP-20260226-000008,null,ZON-007,ZON-103,unfulfilled,PASS,PASS
TRP-20260129-000009,DRV-002591,ZON-119,ZON-004,completed,PASS,PASS
TRP-20260126-000010,DRV-001171,ZON-013,ZON-105,cancelled_by_rider,PASS,PASS


### 12.3 Apply chronology, status, compatibility, distance, fare/surge and lineage checks

The conditions below are null-aware and follow the approved TripPulse contract.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW trips_all_checked AS
SELECT t.*,
  CASE WHEN t.request_ts IS NULL
             OR (t.driver_accept_ts IS NOT NULL AND t.driver_accept_ts < t.request_ts)
             OR (t.pickup_ts IS NOT NULL AND t.driver_accept_ts IS NOT NULL AND t.pickup_ts < t.driver_accept_ts)
             OR (t.pickup_ts IS NOT NULL AND t.driver_accept_ts IS NULL AND t.trip_status IN ('completed'))
             OR (t.trip_status = 'completed' AND t.pickup_ts IS NULL)
             OR (t.trip_status = 'completed' AND (t.dropoff_ts IS NULL OR t.dropoff_ts <= t.pickup_ts))
             OR (t.dropoff_ts IS NOT NULL AND t.pickup_ts IS NOT NULL AND t.dropoff_ts <= t.pickup_ts)
             OR (t.cancel_ts IS NOT NULL AND t.cancel_ts < t.request_ts)
       THEN 'FAIL' ELSE 'PASS' END AS trip003_check,
  CASE WHEN t.trip_status NOT IN ('completed','cancelled_by_rider','cancelled_by_driver','unfulfilled')
             OR (t.trip_status = 'completed' AND (t.cancel_ts IS NOT NULL OR t.cancellation_reason IS NOT NULL OR t.dropoff_ts IS NULL OR t.driver_id IS NULL))
             OR (t.trip_status IN ('cancelled_by_rider','cancelled_by_driver') AND (t.cancel_ts IS NULL OR t.dropoff_ts IS NOT NULL OR t.cancellation_reason IS NULL))
             OR (t.trip_status = 'unfulfilled' AND (t.driver_id IS NOT NULL OR t.pickup_ts IS NOT NULL OR t.dropoff_ts IS NOT NULL OR t.cancel_ts IS NOT NULL))
       THEN 'FAIL' ELSE 'PASS' END AS trip004_check,
  CASE WHEN t.driver_id IS NULL THEN 'PASS'
       WHEN d.driver_id IS NULL OR d.driver_status <> 'active' OR d.service_type <> t.service_type THEN 'FAIL'
       ELSE 'PASS' END AS trip005_check,
  CASE WHEN t.estimated_distance_km IS NULL OR t.estimated_distance_km < 0.30 OR t.estimated_distance_km > 80.00
             OR (t.trip_status = 'completed' AND (t.actual_distance_km IS NULL OR t.actual_distance_km < 0.30 OR t.actual_distance_km > 100.00))
             OR (t.trip_status <> 'completed' AND t.actual_distance_km IS NOT NULL)
       THEN 'FAIL' ELSE 'PASS' END AS trip006_check,
  CASE WHEN t.estimated_fare_inr IS NULL OR t.estimated_fare_inr < 20.00 OR t.estimated_fare_inr > 5000.00
             OR t.surge_multiplier IS NULL OR t.surge_multiplier < 1.00 OR t.surge_multiplier > 3.00
             OR (t.trip_status = 'completed' AND (t.final_fare_inr IS NULL OR t.final_fare_inr < 20.00 OR t.final_fare_inr > 6000.00))
             OR (t.trip_status <> 'completed' AND t.final_fare_inr IS NOT NULL AND t.final_fare_inr <> 0.00)
       THEN 'FAIL' ELSE 'PASS' END AS trip007_check,
  CASE WHEN t.request_ts IS NULL OR t.request_ts < TIMESTAMP '2026-01-01 00:00:00' OR t.request_ts >= TIMESTAMP '2026-04-01 00:00:00'
             OR t.record_created_ts IS NULL
             OR t.record_created_ts < COALESCE(greatest(t.request_ts,t.driver_accept_ts,t.pickup_ts,t.dropoff_ts,t.cancel_ts), t.request_ts)
             OR t.record_created_ts > current_timestamp()
             OR t._bronze_record_hash IS NULL OR t._source_file_name IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS trip008_check
FROM trips_reference_checked t
LEFT JOIN trusted_silver_drivers d ON t.driver_id = d.driver_id;

SELECT trip_id, trip_status, trip003_check, trip004_check, trip005_check, trip006_check, trip007_check, trip008_check
FROM trips_all_checked LIMIT 10;

trip_id,trip_status,trip003_check,trip004_check,trip005_check,trip006_check,trip007_check,trip008_check
TRP-20260127-000001,completed,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260306-000002,completed,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260219-000003,cancelled_by_rider,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260223-000004,completed,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260312-000005,cancelled_by_driver,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260313-000006,unfulfilled,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260228-000007,completed,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260226-000008,unfulfilled,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260129-000009,completed,PASS,PASS,PASS,PASS,PASS,PASS
TRP-20260126-000010,cancelled_by_rider,PASS,PASS,PASS,PASS,PASS,PASS


### 12.4 Capture every failed Trip rule

One physical Trip row can fail several rules. We therefore evaluate every rule independently and combine the results instead of using first-error-only logic.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW trips_dq AS
SELECT *,
  CASE WHEN trip001_check='FAIL' OR trip002_check='FAIL' OR trip003_check='FAIL' OR trip004_check='FAIL' OR trip005_check='FAIL' OR trip006_check='FAIL' OR trip007_check='FAIL' OR trip008_check='FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN trip001_check='FAIL' OR trip002_check='FAIL' OR trip003_check='FAIL' OR trip004_check='FAIL' OR trip005_check='FAIL' OR trip006_check='FAIL' OR trip007_check='FAIL' OR trip008_check='FAIL'
       THEN concat('[', concat_ws(', ',
          CASE WHEN trip001_check='FAIL' THEN 'DQ-TRIP-001' END, CASE WHEN trip002_check='FAIL' THEN 'DQ-TRIP-002' END,
          CASE WHEN trip003_check='FAIL' THEN 'DQ-TRIP-003' END, CASE WHEN trip004_check='FAIL' THEN 'DQ-TRIP-004' END,
          CASE WHEN trip005_check='FAIL' THEN 'DQ-TRIP-005' END, CASE WHEN trip006_check='FAIL' THEN 'DQ-TRIP-006' END,
          CASE WHEN trip007_check='FAIL' THEN 'DQ-TRIP-007' END, CASE WHEN trip008_check='FAIL' THEN 'DQ-TRIP-008' END), ']')
       ELSE '[]' END AS failed_rule_ids,
  concat_ws('; ',
    CASE WHEN trip001_check='FAIL' THEN 'TRIP_KEY_INVALID' END, CASE WHEN trip002_check='FAIL' THEN 'TRIP_REFERENCE_ORPHAN' END,
    CASE WHEN trip003_check='FAIL' THEN 'TRIP_TIMESTAMP_SEQUENCE_INVALID' END, CASE WHEN trip004_check='FAIL' THEN 'TRIP_STATUS_CONDITION_INVALID' END,
    CASE WHEN trip005_check='FAIL' THEN 'TRIP_SERVICE_ASSIGNMENT_INVALID' END, CASE WHEN trip006_check='FAIL' THEN 'TRIP_DISTANCE_INVALID' END,
    CASE WHEN trip007_check='FAIL' THEN 'TRIP_FARE_SURGE_INVALID' END, CASE WHEN trip008_check='FAIL' THEN 'TRIP_WINDOW_OR_LINEAGE_INVALID' END) AS failure_reasons,
  CASE WHEN trip001_check='FAIL' OR trip002_check='FAIL' OR trip003_check='FAIL' THEN 'Critical'
       WHEN trip004_check='FAIL' OR trip005_check='FAIL' OR trip006_check='FAIL' OR trip007_check='FAIL' OR trip008_check='FAIL' THEN 'Major' ELSE 'NONE' END AS severity,
  concat_ws(', ', CASE WHEN trip001_check='FAIL' THEN 'trip_id' END, CASE WHEN trip002_check='FAIL' THEN 'driver_id, pickup_zone_id, dropoff_zone_id' END,
    CASE WHEN trip003_check='FAIL' THEN 'request_ts, driver_accept_ts, pickup_ts, dropoff_ts, cancel_ts' END, CASE WHEN trip004_check='FAIL' THEN 'trip_status, cancel_ts, dropoff_ts, cancellation_reason, driver_id' END,
    CASE WHEN trip005_check='FAIL' THEN 'driver_id, service_type' END, CASE WHEN trip006_check='FAIL' THEN 'estimated_distance_km, actual_distance_km' END,
    CASE WHEN trip007_check='FAIL' THEN 'estimated_fare_inr, final_fare_inr, surge_multiplier' END, CASE WHEN trip008_check='FAIL' THEN 'request_ts, record_created_ts, lineage' END) AS affected_field
FROM trips_all_checked;

SELECT trip_id, trip_status, failed_rule_ids, failure_reasons, severity, dq_status
FROM trips_dq WHERE dq_status='FAIL' LIMIT 20;

trip_id,trip_status,failed_rule_ids,failure_reasons,severity,dq_status
TRP-20260227-000078,completed,"[DQ-TRIP-002, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical,FAIL
TRP-20260119-000117,completed,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical,FAIL
TRP-20260120-000179,cancelled_by_driver,[DQ-TRIP-006],TRIP_DISTANCE_INVALID,Major,FAIL
TRP-20260127-000246,completed,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,FAIL
TRP-20260119-000254,completed,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical,FAIL
TRP-20260224-000288,unfulfilled,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,FAIL
TRP-20260306-000297,completed,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,FAIL
TRP-20260125-000362,completed,"[DQ-TRIP-002, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical,FAIL
TRP-20260227-000412,completed,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,FAIL
TRP-20260103-000414,completed,[DQ-TRIP-001],TRIP_KEY_INVALID,Critical,FAIL


### 12.5 View genuine multi-rule failures


In [0]:
%sql
SELECT trip_id, failed_rule_ids, failure_reasons, severity
FROM trips_dq
WHERE failed_rule_ids LIKE '%,%'
LIMIT 20;

trip_id,failed_rule_ids,failure_reasons,severity
TRP-20260227-000078,"[DQ-TRIP-002, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical
TRP-20260119-000117,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical
TRP-20260119-000254,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical
TRP-20260125-000362,"[DQ-TRIP-002, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical
TRP-20260123-000471,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical
TRP-20260127-000612,"[DQ-TRIP-003, DQ-TRIP-004, DQ-TRIP-006, DQ-TRIP-007]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID; TRIP_DISTANCE_INVALID; TRIP_FARE_SURGE_INVALID,Critical
TRP-20260310-000914,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical
TRP-20260314-001134,"[DQ-TRIP-002, DQ-TRIP-004, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_STATUS_CONDITION_INVALID; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical
TRP-20260313-001261,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical
TRP-20260104-001326,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical


## 13. Build Payment DQ - preserve attempt grain

Payments are one row per **payment attempt**. Group logic is calculated first and then joined back to the affected physical payment rows. We never collapse attempts to trip grain.

### 13.1 Payment key/reference checks


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_payment_ids AS
SELECT payment_id, COUNT(*) AS occurrences
FROM silver_payments_candidate
WHERE payment_id IS NOT NULL AND trim(payment_id) <> ''
GROUP BY payment_id HAVING COUNT(*) > 1;

CREATE OR REPLACE TEMP VIEW duplicate_payment_references AS
SELECT payment_reference, COUNT(*) AS occurrences
FROM silver_payments_candidate
WHERE payment_reference IS NOT NULL AND trim(payment_reference) <> ''
GROUP BY payment_reference HAVING COUNT(*) > 1;

CREATE OR REPLACE TEMP VIEW payments_identity_checked AS
SELECT p.*,
  CASE WHEN p.payment_id IS NULL OR trim(p.payment_id) = '' OR p.payment_id NOT RLIKE '^PAY-[0-9]{9}$'
             OR dpi.payment_id IS NOT NULL
             OR p.payment_reference IS NULL OR p.payment_reference NOT RLIKE '^TPREF-[A-F0-9]{8}$'
             OR dpr.payment_reference IS NOT NULL
             OR p.trip_id IS NULL OR t.trip_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS pay001_check
FROM silver_payments_candidate p
LEFT JOIN duplicate_payment_ids dpi ON p.payment_id = dpi.payment_id
LEFT JOIN duplicate_payment_references dpr ON p.payment_reference = dpr.payment_reference
LEFT JOIN (SELECT DISTINCT trip_id FROM silver_trips_candidate) t ON p.trip_id = t.trip_id;

SELECT payment_id, trip_id, payment_reference, pay001_check FROM payments_identity_checked LIMIT 20;

payment_id,trip_id,payment_reference,pay001_check
PAY-000000001,TRP-20260101-000227,TPREF-00000001,PASS
PAY-000000002,TRP-20260101-000227,TPREF-00000002,PASS
PAY-000000003,TRP-20260101-000445,TPREF-00000003,PASS
PAY-000000004,TRP-20260101-000445,TPREF-00000004,PASS
PAY-000000005,TRP-20260101-000466,TPREF-00000005,PASS
PAY-000000006,TRP-20260101-000466,TPREF-00000006,PASS
PAY-000000007,TRP-20260101-000674,TPREF-00000007,PASS
PAY-000000008,TRP-20260101-000674,TPREF-00000008,PASS
PAY-000000009,TRP-20260101-000760,TPREF-00000009,PASS
PAY-000000010,TRP-20260101-000760,TPREF-0000000A,PASS


### 13.2 Calculate payment-attempt group outcome

The approved rule requires attempt number 1–3, unique `(trip_id, attempt_number)`, valid method/status/reason/amount/time, exactly one final attempt, and successful final-amount reconciliation for completed trips.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payment_group_checks AS
SELECT p.trip_id,
  SUM(CASE WHEN p.attempt_number IS NULL OR p.attempt_number NOT BETWEEN 1 AND 3 THEN 1 ELSE 0 END) AS invalid_attempt_number_rows,
  COUNT(*) AS attempt_rows,
  COUNT(DISTINCT p.attempt_number) AS distinct_attempt_numbers,
  SUM(CASE WHEN p.is_final_attempt THEN 1 ELSE 0 END) AS final_flags,
  SUM(CASE WHEN p.payment_method NOT IN ('upi','card','wallet','cash') OR p.payment_method IS NULL THEN 1 ELSE 0 END) AS invalid_method_rows,
  SUM(CASE WHEN p.payment_status NOT IN ('success','failed','pending','refunded') OR p.payment_status IS NULL THEN 1 ELSE 0 END) AS invalid_status_rows,
  SUM(CASE WHEN p.payment_status='failed' AND (p.failure_reason IS NULL OR p.failure_reason NOT IN ('bank_declined','timeout','insufficient_funds','technical_error')) THEN 1 ELSE 0 END) AS invalid_failure_reason_rows,
  SUM(CASE WHEN p.payment_status<>'failed' AND p.failure_reason IS NOT NULL THEN 1 ELSE 0 END) AS unexpected_failure_reason_rows,
  SUM(CASE WHEN p.amount_inr IS NULL OR p.amount_inr < 0 OR p.amount_inr > 6000 THEN 1 ELSE 0 END) AS invalid_amount_rows,
  SUM(CASE WHEN p.payment_ts IS NULL OR p.payment_ts < t.request_ts OR (t.trip_status='completed' AND p.payment_ts < t.dropoff_ts) THEN 1 ELSE 0 END) AS invalid_time_rows,
  SUM(CASE WHEN p.payment_status='success' AND p.is_final_attempt AND t.trip_status='completed'
                 AND (p.amount_inr IS NULL OR t.final_fare_inr IS NULL OR abs(p.amount_inr - t.final_fare_inr) > 0.01) THEN 1 ELSE 0 END) AS final_amount_mismatch_rows,
  SUM(CASE WHEN p.trip_id IS NOT NULL AND p.attempt_number IS NOT NULL THEN 1 ELSE 0 END) AS keyed_rows
FROM silver_payments_candidate p
LEFT JOIN silver_trips_candidate t ON p.trip_id=t.trip_id
GROUP BY p.trip_id;

CREATE OR REPLACE TEMP VIEW payment_attempt_key_groups AS
SELECT trip_id, attempt_number, COUNT(*) AS occurrences
FROM silver_payments_candidate
GROUP BY trip_id, attempt_number
HAVING COUNT(*) > 1;

CREATE OR REPLACE TEMP VIEW payments_all_checked AS
SELECT p.*,
  CASE WHEN p.pay001_check='FAIL'
             OR g.invalid_attempt_number_rows > 0 OR g.distinct_attempt_numbers <> g.attempt_rows OR g.final_flags <> 1
             OR g.invalid_method_rows > 0 OR g.invalid_status_rows > 0 OR g.invalid_failure_reason_rows > 0 OR g.unexpected_failure_reason_rows > 0
             OR g.invalid_amount_rows > 0 OR g.invalid_time_rows > 0 OR g.final_amount_mismatch_rows > 0
             OR ak.trip_id IS NOT NULL
       THEN 'FAIL' ELSE 'PASS' END AS pay002_check
FROM payments_identity_checked p
LEFT JOIN payment_group_checks g ON p.trip_id=g.trip_id
LEFT JOIN payment_attempt_key_groups ak ON p.trip_id=ak.trip_id AND p.attempt_number=ak.attempt_number;

SELECT payment_id, trip_id, attempt_number, payment_status, is_final_attempt, pay001_check, pay002_check
FROM payments_all_checked LIMIT 30;

payment_id,trip_id,attempt_number,payment_status,is_final_attempt,pay001_check,pay002_check
PAY-000000022,TRP-20260101-001291,2,success,true,PASS,PASS
PAY-000000012,TRP-20260101-000843,2,success,true,PASS,PASS
PAY-000000021,TRP-20260101-001291,1,failed,false,PASS,PASS
PAY-000000011,TRP-20260101-000843,1,failed,false,PASS,PASS
PAY-000000018,TRP-20260101-001086,2,success,true,PASS,PASS
PAY-000000017,TRP-20260101-001086,1,failed,false,PASS,PASS
PAY-000000014,TRP-20260101-000871,2,success,true,PASS,PASS
PAY-000000024,TRP-20260101-001491,2,success,true,PASS,PASS
PAY-000000013,TRP-20260101-000871,1,failed,false,PASS,PASS
PAY-000000023,TRP-20260101-001491,1,failed,false,PASS,PASS


### 13.3 Capture every payment failure


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payments_dq AS
SELECT *,
  CASE WHEN pay001_check='FAIL' OR pay002_check='FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN pay001_check='FAIL' OR pay002_check='FAIL'
       THEN concat('[', concat_ws(', ', CASE WHEN pay001_check='FAIL' THEN 'DQ-PAY-001' END, CASE WHEN pay002_check='FAIL' THEN 'DQ-PAY-002' END), ']')
       ELSE '[]' END AS failed_rule_ids,
  -- FIX: added fallback '[]' for PASS rows (was returning empty string, unlike failed_rule_ids)
  CASE WHEN pay001_check='FAIL' OR pay002_check='FAIL'
       THEN concat('[', concat_ws('; ', CASE WHEN pay001_check='FAIL' THEN 'PAYMENT_KEY_OR_TRIP_INVALID' END, CASE WHEN pay002_check='FAIL' THEN 'PAYMENT_LOGIC_INVALID' END), ']')
       ELSE '[]' END AS failure_reasons,
  -- FIX: when BOTH rules fail, the old CASE returned only 'Critical' (first-match wins).
  --      Now aggregates all severities for consistency with failed_rule_ids / failure_reasons.
  CASE WHEN pay001_check='FAIL' OR pay002_check='FAIL'
       THEN concat('[', concat_ws(', ', CASE WHEN pay001_check='FAIL' THEN 'Critical' END, CASE WHEN pay002_check='FAIL' THEN 'Major' END), ']')
       ELSE '[]' END AS severity,
  -- FIX: same first-match issue — now lists all affected fields when multiple rules fail.
  CASE WHEN pay001_check='FAIL' OR pay002_check='FAIL'
       THEN concat_ws('; ', CASE WHEN pay001_check='FAIL' THEN 'payment_id, payment_reference, trip_id' END, CASE WHEN pay002_check='FAIL' THEN 'attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt' END)
       ELSE '' END AS affected_field
FROM payments_all_checked;

-- NOTE: added affected_field to the preview SELECT for completeness
SELECT payment_id, trip_id, attempt_number, failed_rule_ids, failure_reasons, severity, affected_field, dq_status
FROM payments_dq WHERE dq_status='FAIL' LIMIT 20;

payment_id,trip_id,attempt_number,failed_rule_ids,failure_reasons,severity,affected_field,dq_status
PAY-000111202,TRP-20260221-056289,1,"[DQ-PAY-001, DQ-PAY-002]",[PAYMENT_KEY_OR_TRIP_INVALID; PAYMENT_LOGIC_INVALID],"[Critical, Major]","payment_id, payment_reference, trip_id; attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000102261,TRP-20260216-057712,1,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000127964,TRP-20260302-136906,1,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000046482,TRP-20260115-200460,1,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000003450,TRP-20260101-239209,2,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000137675,TRP-20260308-012579,1,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000157357,TRP-20260319-044198,1,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000088497,TRP-20260208-080665,1,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000102853,TRP-20260216-137278,1,"[DQ-PAY-001, DQ-PAY-002]",[PAYMENT_KEY_OR_TRIP_INVALID; PAYMENT_LOGIC_INVALID],"[Critical, Major]","payment_id, payment_reference, trip_id; attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL
PAY-000016364,TRP-20260105-139200,2,[DQ-PAY-002],[PAYMENT_LOGIC_INVALID],[Major],"attempt_number, payment_status, failure_reason, amount_inr, payment_ts, is_final_attempt",FAIL


## 14. Streaming DQ preparation - four rules, no fabricated Week-10 results

The approved TripPulse rulebook contains four streaming rules. The current Week-5 batch handoff does not contain a streaming Candidate table, so these are documented as the later execution contract. Do **not** create fake event counts in Week 6.

### 14.1 Required streaming classifications

| Rule | Check | Week-10 treatment |
|---|---|---|
| DQ-STR-001 | unique `event_id`; trip/zone/present-driver references resolve | quarantine/duplicate classification |
| DQ-STR-002 | parse `event_ts`; apply 10-minute watermark; retain late events | retained late/invalid classification |
| DQ-STR-003 | increasing `event_sequence_no` and approved transition | quarantine invalid/out-of-order event |
| DQ-STR-004 | explicit schema v1.0, required fields/types/ranges, rescue corrupt/drifted payload | retain raw/rescued payload and source context |


In [0]:
%sql
SELECT 'DQ-STR-001' AS rule_id, 'prepared - event Candidate arrives in Week 10' AS status
UNION ALL SELECT 'DQ-STR-002', 'prepared - 10-minute watermark classification in Week 10'
UNION ALL SELECT 'DQ-STR-003', 'prepared - sequence/state-machine classification in Week 10'
UNION ALL SELECT 'DQ-STR-004', 'prepared - schema/rescue classification in Week 10';

**Boundary:** Week 6 proves batch DQ, retained quarantine, reconciliation and replay guidance. Week 10 proves incremental event processing, checkpointing, watermarking and streaming DQ execution.


## 15. Summarise rule results

### 15.1 Batch routing totals


In [0]:
%sql
SELECT 'zones' AS entity, dq_status, COUNT(*) AS rows FROM zones_routed GROUP BY dq_status
UNION ALL SELECT 'drivers', dq_status, COUNT(*) FROM drivers_routed GROUP BY dq_status
UNION ALL SELECT 'trips', dq_status, COUNT(*) FROM trips_dq GROUP BY dq_status
UNION ALL SELECT 'payments', dq_status, COUNT(*) FROM payments_dq GROUP BY dq_status
ORDER BY entity, dq_status;

entity,dq_status,rows
drivers,FAIL,7
drivers,PASS,2793
payments,FAIL,2680
payments,PASS,177635
trips,FAIL,9221
trips,PASS,241654
zones,PASS,120


### 15.2 Rule-failure scorecard

A rule-failure occurrence can exceed the number of quarantined physical rows because one row may fail multiple rules.


In [0]:
%sql
SELECT 'DQ-ZON-001' AS rule_id, SUM(CASE WHEN zon001_check='FAIL' THEN 1 ELSE 0 END) AS failed_rows FROM zones_routed
UNION ALL SELECT 'DQ-DRV-001', SUM(CASE WHEN drv001_check='FAIL' THEN 1 ELSE 0 END) FROM drivers_routed
UNION ALL SELECT 'DQ-TRIP-001', SUM(CASE WHEN trip001_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-002', SUM(CASE WHEN trip002_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-003', SUM(CASE WHEN trip003_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-004', SUM(CASE WHEN trip004_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-005', SUM(CASE WHEN trip005_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-006', SUM(CASE WHEN trip006_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-007', SUM(CASE WHEN trip007_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-TRIP-008', SUM(CASE WHEN trip008_check='FAIL' THEN 1 ELSE 0 END) FROM trips_dq
UNION ALL SELECT 'DQ-PAY-001', SUM(CASE WHEN pay001_check='FAIL' THEN 1 ELSE 0 END) FROM payments_dq
UNION ALL SELECT 'DQ-PAY-002', SUM(CASE WHEN pay002_check='FAIL' THEN 1 ELSE 0 END) FROM payments_dq
ORDER BY rule_id;

rule_id,failed_rows
DQ-DRV-001,7
DQ-PAY-001,1390
DQ-PAY-002,2680
DQ-TRIP-001,2375
DQ-TRIP-002,2100
DQ-TRIP-003,2000
DQ-TRIP-004,1093
DQ-TRIP-005,1364
DQ-TRIP-006,1252
DQ-TRIP-007,1252


## 16. Write Trusted Silver and Quarantine tables

The outputs retain Candidate columns plus DQ outcome and review metadata.


In [0]:
%sql
CREATE OR REPLACE TABLE trusted_silver_zones USING DELTA AS
SELECT * FROM zones_routed WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_zones USING DELTA AS
SELECT *, current_timestamp() AS quarantined_at, 'P02-W06' AS run_id, 'new' AS rework_status FROM zones_routed WHERE dq_status='FAIL';

CREATE OR REPLACE TABLE trusted_silver_drivers USING DELTA AS
SELECT * FROM drivers_routed WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_drivers USING DELTA AS
SELECT *, current_timestamp() AS quarantined_at, 'P02-W06' AS run_id, 'new' AS rework_status FROM drivers_routed WHERE dq_status='FAIL';

CREATE OR REPLACE TABLE trusted_silver_trips USING DELTA AS
SELECT * FROM trips_dq WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_trips USING DELTA AS
SELECT *, current_timestamp() AS quarantined_at, 'P02-W06' AS run_id, 'new' AS rework_status FROM trips_dq WHERE dq_status='FAIL';

CREATE OR REPLACE TABLE trusted_silver_payments USING DELTA AS
SELECT * FROM payments_dq WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_payments USING DELTA AS
SELECT *, current_timestamp() AS quarantined_at, 'P02-W06' AS run_id, 'new' AS rework_status FROM payments_dq WHERE dq_status='FAIL';

num_affected_rows,num_inserted_rows


**Expected result:** eight batch Delta tables: four Trusted Silver tables and four Quarantine tables. The streaming pair is intentionally not created until streaming Candidate data exists in the approved later phase.


## 17. Inspect useful evidence

### 17.1 Trusted Trip example


In [0]:
%sql
SELECT trip_id, trip_status, driver_id, response_seconds, trip_duration_seconds, dq_status, failed_rule_ids, severity
FROM trusted_silver_trips LIMIT 10;

trip_id,trip_status,driver_id,response_seconds,trip_duration_seconds,dq_status,failed_rule_ids,severity
TRP-20260127-000001,completed,DRV-000620,285,2835,PASS,[],NONE
TRP-20260306-000002,completed,DRV-002433,21,3617,PASS,[],NONE
TRP-20260219-000003,cancelled_by_rider,DRV-002305,170,null,PASS,[],NONE
TRP-20260223-000004,completed,DRV-001102,295,1778,PASS,[],NONE
TRP-20260312-000005,cancelled_by_driver,DRV-000357,313,null,PASS,[],NONE
TRP-20260313-000006,unfulfilled,null,null,null,PASS,[],NONE
TRP-20260228-000007,completed,DRV-002354,90,5155,PASS,[],NONE
TRP-20260226-000008,unfulfilled,null,null,null,PASS,[],NONE
TRP-20260129-000009,completed,DRV-002591,295,3398,PASS,[],NONE
TRP-20260126-000010,cancelled_by_rider,DRV-001171,110,null,PASS,[],NONE


### 17.2 Quarantined Trip example


In [0]:
%sql
SELECT trip_id, trip_status, driver_id, pickup_ts, dropoff_ts, failed_rule_ids, failure_reasons, severity, affected_field, run_id, rework_status
FROM quarantine_trips LIMIT 10;

trip_id,trip_status,driver_id,pickup_ts,dropoff_ts,failed_rule_ids,failure_reasons,severity,affected_field,run_id,rework_status
TRP-20260227-000078,completed,DRV-999999,2026-02-26T18:56:15.000Z,2026-02-26T19:15:58.000Z,"[DQ-TRIP-002, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical,"driver_id, pickup_zone_id, dropoff_zone_id, driver_id, service_type",P02-W06,new
TRP-20260119-000117,completed,DRV-001853,2026-01-19T00:39:12.000Z,null,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical,"request_ts, driver_accept_ts, pickup_ts, dropoff_ts, cancel_ts, trip_status, cancel_ts, dropoff_ts, cancellation_reason, driver_id",P02-W06,new
TRP-20260120-000179,cancelled_by_driver,DRV-000962,null,null,[DQ-TRIP-006],TRIP_DISTANCE_INVALID,Major,"estimated_distance_km, actual_distance_km",P02-W06,new
TRP-20260127-000246,completed,DRV-002651,2026-01-27T05:21:48.000Z,2026-01-27T06:21:13.000Z,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,"request_ts, record_created_ts, lineage",P02-W06,new
TRP-20260119-000254,completed,DRV-001179,2026-01-18T21:51:14.000Z,null,"[DQ-TRIP-003, DQ-TRIP-004]",TRIP_TIMESTAMP_SEQUENCE_INVALID; TRIP_STATUS_CONDITION_INVALID,Critical,"request_ts, driver_accept_ts, pickup_ts, dropoff_ts, cancel_ts, trip_status, cancel_ts, dropoff_ts, cancellation_reason, driver_id",P02-W06,new
TRP-20260224-000288,unfulfilled,null,null,null,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,"request_ts, record_created_ts, lineage",P02-W06,new
TRP-20260306-000297,completed,DRV-000646,2026-03-05T22:44:02.000Z,2026-03-05T23:06:57.000Z,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,"request_ts, record_created_ts, lineage",P02-W06,new
TRP-20260125-000362,completed,DRV-999999,2026-01-25T02:29:39.000Z,2026-01-25T03:30:49.000Z,"[DQ-TRIP-002, DQ-TRIP-005]",TRIP_REFERENCE_ORPHAN; TRIP_SERVICE_ASSIGNMENT_INVALID,Critical,"driver_id, pickup_zone_id, dropoff_zone_id, driver_id, service_type",P02-W06,new
TRP-20260227-000412,completed,DRV-000761,2026-02-27T09:16:09.000Z,2026-02-27T09:48:52.000Z,[DQ-TRIP-008],TRIP_WINDOW_OR_LINEAGE_INVALID,Major,"request_ts, record_created_ts, lineage",P02-W06,new
TRP-20260103-000414,completed,DRV-001532,2026-01-03T14:24:22.000Z,2026-01-03T14:33:59.000Z,[DQ-TRIP-001],TRIP_KEY_INVALID,Critical,trip_id,P02-W06,new


### 17.3 Quarantined Payment example


In [0]:
%sql
SELECT payment_id, trip_id, attempt_number, payment_status, is_final_attempt, failed_rule_ids, failure_reasons, severity
FROM quarantine_payments LIMIT 10;

payment_id,trip_id,attempt_number,payment_status,is_final_attempt,failed_rule_ids,failure_reasons,severity
PAY-000000081,TRP-20260101-003982,1,failed,false,[DQ-PAY-002],PAYMENT_LOGIC_INVALID,Major
PAY-000000082,TRP-20260101-003982,2,success,true,[DQ-PAY-002],PAYMENT_LOGIC_INVALID,Major
PAY-000000107,TRP-20260101-005292,1,failed,false,[DQ-PAY-002],PAYMENT_LOGIC_INVALID,Major
PAY-000000108,TRP-20260101-005292,2,failed,true,[DQ-PAY-002],PAYMENT_LOGIC_INVALID,Major
PAY-000000313,TRP-20260101-020860,1,failed,false,[DQ-PAY-002],PAYMENT_LOGIC_INVALID,Major
PAY-000000314,TRP-20260401-999999,2,success,true,"[DQ-PAY-001, DQ-PAY-002]",PAYMENT_KEY_OR_TRIP_INVALID; PAYMENT_LOGIC_INVALID,Critical
PAY-000000365,TRP-20260101-024738,1,failed,false,[DQ-PAY-002],PAYMENT_LOGIC_INVALID,Major
PAY-000000366,TRP-20260401-999999,2,success,true,"[DQ-PAY-001, DQ-PAY-002]",PAYMENT_KEY_OR_TRIP_INVALID; PAYMENT_LOGIC_INVALID,Critical
PAY-000000455,TRP-20260101-031681,1,failed,false,"[DQ-PAY-001, DQ-PAY-002]",PAYMENT_KEY_OR_TRIP_INVALID; PAYMENT_LOGIC_INVALID,Critical
PAY-000000456,TRP-20260101-031681,2,success,true,"[DQ-PAY-001, DQ-PAY-002]",PAYMENT_KEY_OR_TRIP_INVALID; PAYMENT_LOGIC_INVALID,Critical


## 18. Prove no silent loss

For every batch entity:

`Candidate rows = Trusted rows + Quarantine rows`

### 18.1 Count reconciliation


In [0]:
%sql
SELECT 'zones' AS entity,
  (SELECT COUNT(*) FROM silver_zones_candidate) AS candidate,
  (SELECT COUNT(*) FROM trusted_silver_zones) AS trusted,
  (SELECT COUNT(*) FROM quarantine_zones) AS quarantine,
  (SELECT COUNT(*) FROM silver_zones_candidate) - (SELECT COUNT(*) FROM trusted_silver_zones) - (SELECT COUNT(*) FROM quarantine_zones) AS variance
UNION ALL SELECT 'drivers',
  (SELECT COUNT(*) FROM silver_drivers_candidate),(SELECT COUNT(*) FROM trusted_silver_drivers),(SELECT COUNT(*) FROM quarantine_drivers),
  (SELECT COUNT(*) FROM silver_drivers_candidate)-(SELECT COUNT(*) FROM trusted_silver_drivers)-(SELECT COUNT(*) FROM quarantine_drivers)
UNION ALL SELECT 'trips',
  (SELECT COUNT(*) FROM silver_trips_candidate),(SELECT COUNT(*) FROM trusted_silver_trips),(SELECT COUNT(*) FROM quarantine_trips),
  (SELECT COUNT(*) FROM silver_trips_candidate)-(SELECT COUNT(*) FROM trusted_silver_trips)-(SELECT COUNT(*) FROM quarantine_trips)
UNION ALL SELECT 'payments',
  (SELECT COUNT(*) FROM silver_payments_candidate),(SELECT COUNT(*) FROM trusted_silver_payments),(SELECT COUNT(*) FROM quarantine_payments),
  (SELECT COUNT(*) FROM silver_payments_candidate)-(SELECT COUNT(*) FROM trusted_silver_payments)-(SELECT COUNT(*) FROM quarantine_payments);

entity,candidate,trusted,quarantine,variance
zones,120,120,0,0
drivers,2800,2793,7,0
trips,250875,241654,9221,0
payments,180315,177635,2680,0


In [0]:
%sql
-- Recreate payment tables with corrected DQ results (no duplicates)
CREATE OR REPLACE TABLE trusted_silver_payments USING DELTA AS
SELECT * FROM payments_dq WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_payments USING DELTA AS
SELECT *, current_timestamp() AS quarantined_at, 'P02-W06' AS run_id, 'new' AS rework_status 
FROM payments_dq WHERE dq_status='FAIL';

num_affected_rows,num_inserted_rows


**Pass condition:** every variance is `0`.

Count equality is necessary, but it does not alone prove the same physical records were routed. The next checks use `_bronze_record_hash`.


### 18.2 Physical-record membership proof - Trips


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW trip_route_membership AS
SELECT _bronze_record_hash, COUNT(*) AS route_occurrences
FROM (
  SELECT _bronze_record_hash FROM trusted_silver_trips
  UNION ALL
  SELECT _bronze_record_hash FROM quarantine_trips
)
GROUP BY _bronze_record_hash;

SELECT COUNT(*) AS candidate_hashes_not_routed_once
FROM silver_trips_candidate c
LEFT JOIN trip_route_membership r ON c._bronze_record_hash=r._bronze_record_hash
WHERE r.route_occurrences IS NULL OR r.route_occurrences <> 1;

candidate_hashes_not_routed_once
1750


**Expected result:** `0`. Repeat the same membership pattern for Zones, Drivers and Payments when capturing final evidence.


## 19. Controlled rerun test

1. Save the current reconciliation result.
2. Rerun the helper-view, checked-view, routed-view and table-write cells in order.
3. Run the reconciliation and membership checks again.

**Pass condition:** the same Candidate snapshot produces the same Trusted/Quarantine membership and counts, with no extra physical records. Execution timestamps may change; business membership must remain stable.

Do not demonstrate rerun safety by deleting outputs manually.


In [0]:
%sql
SELECT 'rerun_check' AS check_name,
       (SELECT COUNT(*) FROM silver_trips_candidate) AS candidate_trips,
       (SELECT COUNT(*) FROM trusted_silver_trips) AS trusted_trips,
       (SELECT COUNT(*) FROM quarantine_trips) AS quarantine_trips,
       (SELECT COUNT(*) FROM silver_trips_candidate)-(SELECT COUNT(*) FROM trusted_silver_trips)-(SELECT COUNT(*) FROM quarantine_trips) AS variance;

check_name,candidate_trips,trusted_trips,quarantine_trips,variance
rerun_check,250875,241654,9221,0


## 20. Exit checklist

- [ ] I can explain Candidate, Trusted Silver and Quarantine in my own words.
- [ ] All four Week-5 batch Candidate tables exist and starting counts were recorded.
- [ ] DQ-ZON-001 and DQ-DRV-001 are implemented in dependency order.
- [ ] DQ-TRIP-001 through DQ-TRIP-008 are implemented.
- [ ] DQ-PAY-001 and DQ-PAY-002 are implemented without collapsing payment attempts.
- [ ] DQ-STR-001 through DQ-STR-004 are represented as later streaming-preparation rules.
- [ ] Every batch rule is visible as a `PASS/FAIL` result.
- [ ] Multi-rule failures retain every applicable reason in one physical row.
- [ ] Four Trusted Silver and four Quarantine Delta tables were created.
- [ ] Candidate = Trusted + Quarantine passes for every batch entity.
- [ ] Physical-record membership proof returns zero exceptions.
- [ ] Controlled rerun evidence is recorded.
- [ ] Correction and replay is documented without direct Quarantine-to-Trusted promotion.
- [ ] GitHub commits, Week Log and AI note are current.

**Week-6 completion standard:** Gold reads only from governed Trusted Silver after these checks pass. Gold/KPI work is Week 7; controlled streaming execution is Week 10.


### Final boundary

**Completed here:** batch DQ rules, failure context, severity, Trusted/Quarantine routing, payment group validation, reconciliation, rerun and replay guidance.  
**Next:** governed Gold design and analytical outputs only after Trusted Silver is accepted.  
**Later:** streaming event deduplication, watermark and event-time quality checks.

> Build. Prove. Present. A trusted table is not trusted because of its name; it is trusted because its rules, evidence and reconciliation are explainable.